
##Week 3-B: Gradio UI 統合セル
   - upload → click → track（既存 Step 4）
   - 指標計算（compute_all_metrics）
   - 5パネル matplotlib を gr.Plot で表示
   - 消失閾値 / 最小成分サイズのスライダー（再追跡なしで再計算）
   - masks.npz / metrics.json / metrics.png / tracked.mp4 のダウンロード

###前提:
   - 既存セルで `predictor` がビルド済み（GPU）
   - ffmpeg がインストール済み（Colab はデフォルト）



In [ ]:

import os
import json
import subprocess
import tempfile

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
import cv2
from PIL import Image
import gradio as gr


# --- 指標関数（src/metrics.py の内容を Colab セルに self-contain）-------------

def mask_area_change_rate(masks):
    areas = masks.sum(axis=(1, 2)).astype(np.float64)
    change_rate = np.zeros(len(areas), dtype=np.float32)
    for t in range(1, len(areas)):
        prev, curr = areas[t - 1], areas[t]
        if prev == 0:
            change_rate[t] = 0.0 if curr == 0 else float('inf')
        else:
            change_rate[t] = (curr - prev) / prev
    return change_rate


def connected_components_count(masks, min_size=50):
    counts = np.zeros(len(masks), dtype=np.int32)
    for t in range(len(masks)):
        if masks[t].sum() == 0:
            counts[t] = 0
            continue
        labeled, n = ndimage.label(masks[t])
        if n == 0:
            counts[t] = 0
        elif min_size > 0:
            sizes = ndimage.sum(masks[t], labeled, range(1, n + 1))
            counts[t] = int((sizes >= min_size).sum())
        else:
            counts[t] = n
    return counts


def largest_component_ratio(masks):
    ratios = np.ones(len(masks), dtype=np.float32)
    for t in range(len(masks)):
        total = masks[t].sum()
        if total == 0:
            ratios[t] = 1.0
            continue
        labeled, n = ndimage.label(masks[t])
        if n == 0:
            ratios[t] = 1.0
            continue
        sizes = ndimage.sum(masks[t], labeled, range(1, n + 1))
        ratios[t] = float(sizes.max() / total)
    return ratios


def shape_iou_temporal(masks):
    T = len(masks)
    iou = np.ones(T, dtype=np.float32)
    if T == 0:
        return iou
    H, W = masks[0].shape

    def _centroid(m):
        ys, xs = np.where(m)
        if len(ys) == 0:
            return None
        return float(ys.mean()), float(xs.mean())

    for t in range(1, T):
        m_prev, m_curr = masks[t - 1], masks[t]
        if m_prev.sum() == 0 or m_curr.sum() == 0:
            iou[t] = 1.0
            continue
        c_prev = _centroid(m_prev)
        c_curr = _centroid(m_curr)
        dy = int(round(c_prev[0] - c_curr[0]))
        dx = int(round(c_prev[1] - c_curr[1]))
        shifted = np.zeros_like(m_curr)
        ys, ye = max(0, dy), min(H, H + dy)
        xs, xe = max(0, dx), min(W, W + dx)
        if ys < ye and xs < xe:
            shifted[ys:ye, xs:xe] = m_curr[ys - dy:ye - dy, xs - dx:xe - dx]
        inter = np.logical_and(m_prev, shifted).sum()
        union = np.logical_or(m_prev, shifted).sum()
        iou[t] = float(inter / union) if union > 0 else 1.0
    return iou


def mask_disappearance(masks, threshold_abs=50, threshold_rel=0.01, ref_frames=5):
    areas = masks.sum(axis=(1, 2)).astype(np.float64)
    ref = areas[:max(ref_frames, 1)]
    ref_area = float(ref.mean()) if len(ref) > 0 else 1.0
    threshold = max(float(threshold_abs), ref_area * threshold_rel)
    is_dis = areas < threshold
    cons = np.zeros(len(areas), dtype=np.int32)
    for t in range(len(areas)):
        if is_dis[t]:
            cons[t] = (cons[t - 1] if t > 0 else 0) + 1
        else:
            cons[t] = 0
    return {
        'areas': areas,
        'is_disappeared': is_dis,
        'consecutive': cons,
        'threshold': threshold,
        'reference_area': ref_area,
    }


def compute_all_metrics(masks, threshold_abs=50, min_size=50):
    area_change = mask_area_change_rate(masks)
    components = connected_components_count(masks, min_size=min_size)
    largest = largest_component_ratio(masks)
    disappear = mask_disappearance(masks, threshold_abs=threshold_abs)
    shape_iou = shape_iou_temporal(masks)
    return {
        'frame_count': int(len(masks)),
        'areas': disappear['areas'].astype(float).tolist(),
        'area_change_rate': area_change.tolist(),
        'connected_components': components.tolist(),
        'largest_component_ratio': largest.tolist(),
        'shape_iou_temporal': shape_iou.tolist(),
        'is_disappeared': disappear['is_disappeared'].tolist(),
        'consecutive_disappearance': disappear['consecutive'].tolist(),
        'reference_area': float(disappear['reference_area']),
        'disappearance_threshold': float(disappear['threshold']),
        'min_component_size': int(min_size),
    }


# --- 5パネル可視化 ----------------------------------------------------------

def make_5panel_plot(metrics, case_name='session'):
    plt.rcParams['font.family'] = 'DejaVu Sans'
    fig, axes = plt.subplots(5, 1, figsize=(11, 12), sharex=True)
    frames = np.arange(metrics['frame_count'])

    # 0. Mask area + threshold
    axes[0].plot(frames, metrics['areas'], color='steelblue', linewidth=1.5)
    axes[0].axhline(
        metrics['disappearance_threshold'], color='red', linestyle='--',
        label=f"Disappearance threshold ({metrics['disappearance_threshold']:.0f}px)",
    )
    axes[0].set_ylabel('Mask area (px)')
    axes[0].set_title(f'{case_name} - mask area over time')
    axes[0].legend(loc='upper right')
    axes[0].grid(alpha=0.3)

    # 1. Area change rate
    rates = np.array(metrics['area_change_rate'], dtype=float)
    rates_disp = np.clip(rates, -2.0, 2.0)
    axes[1].plot(frames, rates_disp, color='darkorange', linewidth=1.5)
    axes[1].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='+-50%')
    axes[1].axhline(-0.5, color='red', linestyle='--', alpha=0.5)
    axes[1].set_ylabel('Area change rate')
    axes[1].set_title('Indicator 1: Mask area change rate (clipped to +-2.0)')
    axes[1].legend(loc='upper right')
    axes[1].grid(alpha=0.3)

    # 2. Components
    axes[2].plot(
        frames, metrics['connected_components'],
        color='seagreen', linewidth=1.5, marker='o', markersize=3,
    )
    axes[2].set_ylabel('Component count')
    axes[2].set_title(
        f'Indicator 2: Connected components (>={metrics["min_component_size"]}px only)'
    )
    axes[2].grid(alpha=0.3)

    # 3. Shape IoU
    siou = np.array(metrics['shape_iou_temporal'], dtype=float)
    axes[3].plot(frames, siou, color='purple', linewidth=1.5)
    axes[3].axhline(0.8, color='red', linestyle='--', alpha=0.5, label='IoU 0.8')
    axes[3].set_ylim(0, 1.05)
    min_idx = int(np.argmin(siou))
    axes[3].set_title(
        f'Indicator 3: Shape stability (centroid-aligned IoU, min '
        f'{float(siou.min()):.3f} at frame {min_idx})'
    )
    axes[3].set_ylabel('Shape IoU')
    axes[3].legend(loc='lower right')
    axes[3].grid(alpha=0.3)

    # 4. Disappearance
    is_dis = np.array(metrics['is_disappeared'])
    cons = np.array(metrics['consecutive_disappearance'])
    axes[4].fill_between(
        frames, 0, is_dis.astype(int),
        alpha=0.4, color='crimson', step='post', label='Disappeared',
    )
    if cons.max() > 0:
        axes[4].plot(
            frames, cons / max(cons.max(), 1),
            color='crimson', linewidth=1.5, label='Consecutive (normalized)',
        )
    axes[4].set_ylim(-0.05, 1.1)
    axes[4].set_ylabel('Disappearance')
    axes[4].set_title(
        f'Indicator 4: Mask disappearance (max consecutive: {int(cons.max())} frames)'
    )
    axes[4].set_xlabel('Frame')
    axes[4].legend(loc='upper left')
    axes[4].grid(alpha=0.3)

    plt.tight_layout()
    return fig


# --- Gradio コールバック ----------------------------------------------------

def upload_video(video_path, state):
    if video_path is None:
        return None, state, 'Upload a video first'
    out_dir = tempfile.mkdtemp(prefix='gradio_frames_')
    cmd = [
        'ffmpeg', '-y', '-i', video_path,
        '-vf', 'fps=10,pad=ceil(iw/2)*2:ceil(ih/2)*2',
        '-q:v', '2', os.path.join(out_dir, '%05d.jpg'),
    ]
    subprocess.run(cmd, check=True, capture_output=True)
    frames = sorted(os.listdir(out_dir))
    if not frames:
        return None, state, 'No frames extracted'
    first = np.array(Image.open(os.path.join(out_dir, frames[0])))

    inference_state = predictor.init_state(video_path=out_dir)

    state = {
        'frame_dir': out_dir,
        'video_path': video_path,
        'n_frames': len(frames),
        'first_frame': first,
        'click_xy': None,
        'masks': None,
        'tracked_mp4': None,
        'masks_npz': None,
        'metrics_json': None,
        'metrics_png': None,
        'out_dir': None,
        'inference_state': inference_state,
    }
    return first, state, f'Loaded {len(frames)} frames. Click on the target in the image.'


def on_click(state, evt: gr.SelectData):
    if state is None or 'first_frame' not in state:
        return None, state, 'Upload a video first'
    x, y = int(evt.index[0]), int(evt.index[1])
    state['click_xy'] = (x, y)
    img = state['first_frame'].copy()
    cv2.drawMarker(img, (x, y), (255, 0, 0),
                   markerType=cv2.MARKER_CROSS, markerSize=30, thickness=3)
    return img, state, f'Clicked at ({x}, {y}). Press "Run tracking".'


def run_tracking(state, thr_abs, min_comp, progress=gr.Progress()):
    if state is None or state.get('click_xy') is None:
        return None, None, None, None, None, state, 'Click a point first'

    inference_state = state['inference_state']
    predictor.reset_state(inference_state)
    x, y = state['click_xy']

    predictor.add_new_points_or_box(
        inference_state=inference_state, frame_idx=0, obj_id=1,
        points=np.array([[x, y]], dtype=np.float32),
        labels=np.array([1], dtype=np.int32),
    )

    frame_dir = state['frame_dir']
    frames_list = sorted(os.listdir(frame_dir))
    H, W = state['first_frame'].shape[:2]
    masks_all = np.zeros((len(frames_list), H, W), dtype=bool)

    progress(0, desc='Propagating')
    for fidx, _obj_ids, mask_logits in predictor.propagate_in_video(inference_state):
        if fidx < len(frames_list):
            m = (mask_logits[0] > 0.0).cpu().numpy()[0]
            masks_all[fidx] = m
        progress((fidx + 1) / len(frames_list), desc=f'Frame {fidx+1}/{len(frames_list)}')

    # オーバーレイ MP4
    out_dir = tempfile.mkdtemp(prefix='gradio_out_')
    overlay_dir = os.path.join(out_dir, 'overlay')
    os.makedirs(overlay_dir, exist_ok=True)
    for i, fname in enumerate(frames_list):
        img = cv2.imread(os.path.join(frame_dir, fname))
        if masks_all[i].any():
            overlay = img.copy()
            overlay[masks_all[i]] = (0, 255, 0)
            img = cv2.addWeighted(img, 0.6, overlay, 0.4, 0)
        # クリック点をすべてのフレームに表示
        cv2.drawMarker(img, (x, y), (0, 0, 255),
                       markerType=cv2.MARKER_CROSS, markerSize=20, thickness=2)
        cv2.imwrite(os.path.join(overlay_dir, f'{i:05d}.jpg'), img)

    tracked_mp4 = os.path.join(out_dir, 'tracked.mp4')
    subprocess.run([
        'ffmpeg', '-y', '-framerate', '10',
        '-i', os.path.join(overlay_dir, '%05d.jpg'),
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', tracked_mp4,
    ], check=True, capture_output=True)

    # masks.npz
    masks_npz = os.path.join(out_dir, 'masks.npz')
    np.savez_compressed(
        masks_npz, masks=masks_all,
        click_xy=np.array([x, y]), case_name='session',
    )

    # metrics
    metrics = compute_all_metrics(
        masks_all, threshold_abs=int(thr_abs), min_size=int(min_comp),
    )
    metrics_json = os.path.join(out_dir, 'metrics.json')
    with open(metrics_json, 'w') as f:
        json.dump(metrics, f, indent=2)

    fig = make_5panel_plot(metrics, case_name='session')
    metrics_png = os.path.join(out_dir, 'metrics.png')
    fig.savefig(metrics_png, dpi=120, bbox_inches='tight')

    state['masks'] = masks_all
    state['tracked_mp4'] = tracked_mp4
    state['masks_npz'] = masks_npz
    state['metrics_json'] = metrics_json
    state['metrics_png'] = metrics_png
    state['out_dir'] = out_dir

    n_with_mask = int((masks_all.sum(axis=(1, 2)) > 0).sum())
    siou_arr = np.array(metrics['shape_iou_temporal'], dtype=float)
    summary = (
        f'Done: {n_with_mask}/{len(frames_list)} frames with mask. '
        f'Click: ({x}, {y}). '
        f'Min shape IoU: {float(siou_arr.min()):.3f} at frame {int(np.argmin(siou_arr))}. '
        f'Disappeared frames: {sum(metrics["is_disappeared"])}, '
        f'max consecutive: {max(metrics["consecutive_disappearance"])}. '
        f'Output dir: {out_dir}'
    )
    return tracked_mp4, fig, masks_npz, metrics_json, metrics_png, state, summary


def recompute_metrics(state, thr_abs, min_comp):
    if state is None or state.get('masks') is None:
        return None, None, None, 'Run tracking first'
    masks_all = state['masks']
    metrics = compute_all_metrics(
        masks_all, threshold_abs=int(thr_abs), min_size=int(min_comp),
    )
    out_dir = state['out_dir']
    metrics_json = os.path.join(out_dir, 'metrics.json')
    with open(metrics_json, 'w') as f:
        json.dump(metrics, f, indent=2)
    fig = make_5panel_plot(metrics, case_name='session')
    metrics_png = os.path.join(out_dir, 'metrics.png')
    fig.savefig(metrics_png, dpi=120, bbox_inches='tight')
    state['metrics_json'] = metrics_json
    state['metrics_png'] = metrics_png
    summary = (
        f'Recomputed. Threshold={int(thr_abs)}px, min_comp={int(min_comp)}px. '
        f'Disappeared frames: {sum(metrics["is_disappeared"])}, '
        f'max consecutive: {max(metrics["consecutive_disappearance"])}.'
    )
    return fig, metrics_json, metrics_png, summary


# --- UI ---------------------------------------------------------------------

with gr.Blocks(title='SAM2 Recognition Limit Observer') as demo:
    gr.Markdown(
        '# SAM2 Recognition Limit Observer\n'
        '車載カメラ映像に SAM2 動画追跡を適用し、'
        'マスクの破綻を 4 指標で可視化します。'
    )
    state = gr.State(value=None)

    with gr.Row():
        with gr.Column(scale=1):
            video_in = gr.Video(label='Input video', sources=['upload'])
            upload_btn = gr.Button('1. Load video', variant='secondary')
            first_frame = gr.Image(
                label='2. Click on the target',
                type='numpy', interactive=False,
            )
        with gr.Column(scale=1):
            thr_abs = gr.Slider(
                0, 1000, value=50, step=10,
                label='Disappearance threshold (abs px)',
            )
            min_comp = gr.Slider(
                0, 500, value=50, step=10,
                label='Min component size (px)',
            )
            track_btn = gr.Button('3. Run tracking', variant='primary')
            recompute_btn = gr.Button(
                '4. Recompute metrics (no re-track)', variant='secondary',
            )
            tracked_video = gr.Video(label='Tracked output')

    status = gr.Textbox(label='Status', value='Ready', lines=2)
    metrics_plot = gr.Plot(label='Metrics (5 panels)')

    with gr.Row():
        dl_mp4 = gr.File(label='tracked.mp4', visible=True)
        dl_npz = gr.File(label='masks.npz', visible=True)
        dl_json = gr.File(label='metrics.json', visible=True)
        dl_png = gr.File(label='metrics.png', visible=True)

    upload_btn.click(
        upload_video, [video_in, state],
        [first_frame, state, status],
    )
    first_frame.select(
        on_click, [state],
        [first_frame, state, status],
    )
    track_btn.click(
        run_tracking, [state, thr_abs, min_comp],
        [tracked_video, metrics_plot, dl_npz, dl_json, dl_png, state, status],
    ).then(
        lambda s: s.get('tracked_mp4') if s else None,
        [state], [dl_mp4],
    )
    recompute_btn.click(
        recompute_metrics, [state, thr_abs, min_comp],
        [metrics_plot, dl_json, dl_png, status],
    )

demo.launch(share=True, debug=False)